# RAG Pipeline: Claim Verification + Narrative Framing Detection

**Pipeline:**
1. Load knowledge base from `rag_data/` (tiered JSONL files)
2. Build FAISS vector index with `paraphrase-multilingual-MiniLM-L12-v2`
3. Extract atomic claims from LLM answer (via fine-tuned Qwen or zero-shot)
4. For each claim: retrieve top-3 evidence snippets (tier-weighted)
5. Verify each claim with GPT-4o-mini → SUPPORTED / NOT SUPPORTED / INSUFFICIENT
6. **Narrative Framing Detection**: compare how different LLMs describe the same topic
7. Generate final human-readable summary

In [14]:
!pip install sentence-transformers faiss-cpu openai tqdm -q


[notice] A new release of pip is available: 26.0.1 -> 26.1
[notice] To update, run: pip install --upgrade pip


## Imports & Config

In [ ]:
import os
import json
import pickle
import numpy as np
from pathlib import Path
from tqdm import tqdm
from collections import defaultdict

import faiss
from sentence_transformers import SentenceTransformer
from openai import OpenAI

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY", "YOUR_KEY_HERE")
RAG_DATA_DIR = Path("../rag_data")
INDEX_CACHE = Path("rag_index.pkl")
EMBED_MODEL = "paraphrase-multilingual-MiniLM-L12-v2"
TOP_K = 3

TIER_WEIGHTS = {1: 1.0, 2: 0.95, 3: 0.85, 4: 0.80, 5: 0.75, 6: 0.90, 7: 0.70, 8: 0.60}

client = OpenAI(api_key=OPENAI_API_KEY)
embedder = SentenceTransformer(EMBED_MODEL)

print("Config loaded")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 6034.14it/s]
BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Config loaded


## Load Knowledge Base

In [ ]:
def load_knowledge_base(data_dir: Path) -> list[dict]:
    docs = []
    jsonl_files = list(data_dir.rglob("*.jsonl"))
    print(f"Found {len(jsonl_files)} JSONL files")
    
    for path in jsonl_files:
        with open(path, encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                try:
                    doc = json.loads(line)
                    if len(doc.get("text", "")) > 50:
                        docs.append(doc)
                except json.JSONDecodeError:
                    continue

    tier_counts = defaultdict(int)
    for doc in docs:
        tier_counts[doc.get("tier", "?")]
        tier_counts[doc.get("tier", "?")] += 1
    
    print(f"\nTotal documents: {len(docs)}")
    for tier in sorted(tier_counts):
        print(f"  Tier {tier}: {tier_counts[tier]} docs")
    
    return docs


def chunk_doc(doc: dict, chunk_size: int = 400) -> list[dict]:
    text = doc["text"]
    words = text.split()
    chunks = []
    
    if len(words) <= chunk_size:
        return [doc]
    
    step = chunk_size - 50
    for i in range(0, len(words), step):
        chunk_text = " ".join(words[i:i + chunk_size])
        chunk = doc.copy()
        chunk["text"] = chunk_text
        chunk["chunk_id"] = f"{doc['id']}_{i}"
        chunks.append(chunk)
    
    return chunks

raw_docs = load_knowledge_base(RAG_DATA_DIR)

all_chunks = []
for doc in tqdm(raw_docs, desc="Chunking docs"):
    all_chunks.extend(chunk_doc(doc))

print(f"\nTotal chunks after splitting: {len(all_chunks)}")

Found 9 JSONL files

Total documents: 999
  Tier 2: 30 docs
  Tier 3: 20 docs
  Tier 4: 20 docs
  Tier 5: 143 docs
  Tier 6: 20 docs
  Tier 7: 144 docs
  Tier 8: 622 docs


Chunking docs: 100%|██████████| 999/999 [00:00<00:00, 18560.84it/s]


Total chunks after splitting: 2747


## Build FAISS Index (with caching)

In [ ]:
def build_index(chunks: list[dict], cache_path: Path):
    
    if cache_path.exists():
        print(f"Loading cached index from {cache_path}...")
        with open(cache_path, "rb") as f:
            data = pickle.load(f)
        return data["index"], data["chunks"], data["embeddings"]
    
    print(f"Building index for {len(chunks)} chunks...")
    texts = [c["text"] for c in chunks]

    embeddings = embedder.encode(
        texts,
        batch_size=64,
        show_progress_bar=True,
        normalize_embeddings=True
    )
    dim = embeddings.shape[1]
    index = faiss.IndexFlatIP(dim)
    index.add(embeddings.astype(np.float32))

    with open(cache_path, "wb") as f:
        pickle.dump({"index": index, "chunks": chunks, "embeddings": embeddings}, f)
    print(f"✅ Index saved to {cache_path}")
    
    return index, chunks, embeddings


index, chunks, embeddings = build_index(all_chunks, INDEX_CACHE)
print(f"\nIndex ready: {index.ntotal} vectors, dim={embeddings.shape[1]}")

Building index for 2747 chunks...


Batches: 100%|██████████| 43/43 [00:06<00:00,  6.45it/s]


✅ Index saved to rag_index.pkl

Index ready: 2747 vectors, dim=384


## Retrieval with Tier-Weighted Re-ranking

In [ ]:
def retrieve(claim: str, top_k: int = TOP_K, fetch_k: int = 20) -> list[dict]:
    query_vec = embedder.encode([claim], normalize_embeddings=True).astype(np.float32)
    
    scores, indices = index.search(query_vec, fetch_k)
    scores, indices = scores[0], indices[0]
    
    candidates = []
    seen_ids = set()
    
    for score, idx in zip(scores, indices):
        if idx == -1:
            continue
        chunk = chunks[idx]
        doc_id = chunk.get("id", str(idx))
        
        if doc_id in seen_ids:
            continue
        seen_ids.add(doc_id)
        
        tier = chunk.get("tier", 5)
        tier_weight = TIER_WEIGHTS.get(tier, 0.7)
        final_score = float(score) * tier_weight
        
        candidates.append({
            "text":         chunk["text"][:600],
            "source":       chunk.get("source", ""),
            "tier":         tier,
            "url":          chunk.get("url", ""),
            "title":        chunk.get("title", ""),
            "date":         chunk.get("date", ""),
            "cosine_score": float(score),
            "final_score":  final_score,
        })
    
    candidates.sort(key=lambda x: x["final_score"], reverse=True)
    return candidates[:top_k]


test_claim = "Велика Британія надала Україні летальну зброю у 2014 році."
evidence = retrieve(test_claim)
print(f"Evidence for: '{test_claim}'\n")
for i, e in enumerate(evidence, 1):
    print(f"[{i}] Tier {e['tier']} | Score: {e['final_score']:.3f} | Source: {e['source']}")
    print(f"    {e['text'][:200]}...\n")

Evidence for: 'Велика Британія надала Україні летальну зброю у 2014 році.'

[1] Tier 8 | Score: 0.373 | Source: Wikipedia EN
    decision and praised current ongoing military training programs between both countries. Russian invasion of Ukraine Following the Russian invasion of Ukraine, the UK provided Ukraine substantial suppo...

[2] Tier 8 | Score: 0.347 | Source: Wikipedia EN
    Minister Denys Shmyhal announced Ukraine had received more than $12 billion worth of weapons and financial aid from Western countries since the start of Russia's invasion on 24 February. On 10 May, th...

[3] Tier 8 | Score: 0.346 | Source: Wikipedia EN
    Many entities have provided or promised military aid to Ukraine during the Russo-Ukrainian War, particularly since the 2022 Russian invasion of Ukraine. This includes weaponry, equipment, training, lo...



## Claim Extraction

In production: use your fine-tuned Qwen model.  
Here: zero-shot GPT-4o-mini as fallback (same format as Qwen output).

In [ ]:
def extract_claims_zeroshot(text: str) -> list[str]:
    prompt = """Витягни атомарні твердження (claims) з тексту нижче.
Кожне твердження має бути одним простим реченням, яке можна перевірити окремо.
Поверни ТІЛЬКИ JSON у форматі: {"claims": ["твердження 1", "твердження 2", ...]}
Без коментарів, без markdown.

Текст:
"""
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt + text}],
        temperature=0,
        response_format={"type": "json_object"}
    )
    
    result = json.loads(response.choices[0].message.content)
    return result.get("claims", [])


def extract_claims_qwen(text: str, model) -> list[str]:
    # inputs = tokenizer(text, return_tensors="pt").to(model.device)
    # output = model.generate(**inputs, max_new_tokens=512)
    # decoded = tokenizer.decode(output[0], skip_special_tokens=True)
    # result = json.loads(decoded)
    # return result.get("claims", [])
    raise NotImplementedError("Connect your Qwen model here")

test_answer = """У 2014 році представники Великої Британії активно підтримували суверенітет України, 
засуджуючи анексію Криму Росією. Вони заявляли про готовність допомогти Україні в забезпеченні 
її територіальної цілісності та закликали до міжнародної підтримки українського уряду."""

claims = extract_claims_zeroshot(test_answer)
print(f"Extracted {len(claims)} claims:")
for i, c in enumerate(claims, 1):
    print(f"  {i}. {c}")

Extracted 4 claims:
  1. У 2014 році представники Великої Британії активно підтримували суверенітет України.
  2. Представники Великої Британії засуджували анексію Криму Росією.
  3. Представники Великої Британії заявляли про готовність допомогти Україні в забезпеченні її територіальної цілісності.
  4. Представники Великої Британії закликали до міжнародної підтримки українського уряду.


## Claim Verification

In [ ]:
VERIFY_PROMPT = """Ти — верифікатор фактів. Твоє завдання — перевірити твердження на основі наданих доказів.

Твердження: {claim}

Докази з бази знань:
{evidence}

На основі ТІЛЬКИ наведених доказів (не власних знань) визнач:
- SUPPORTED: докази підтверджують твердження
- NOT_SUPPORTED: докази суперечать твердженню
- INSUFFICIENT: доказів недостатньо для висновку

Поверни JSON: {{"label": "SUPPORTED"|"NOT_SUPPORTED"|"INSUFFICIENT", "reasoning": "коротке пояснення"}}"""


def verify_claim(claim: str, evidence_list: list[dict]) -> dict:
    
    evidence_text = ""
    for i, e in enumerate(evidence_list, 1):
        evidence_text += f"[{i}] (Tier {e['tier']}, {e['source']}, {e['date']})\n{e['text']}\n\n"
    
    prompt = VERIFY_PROMPT.format(claim=claim, evidence=evidence_text.strip())
    
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=0,
        response_format={"type": "json_object"}
    )
    
    result = json.loads(response.choices[0].message.content)
    
    return {
        "claim":     claim,
        "label":     result.get("label", "INSUFFICIENT"),
        "reasoning": result.get("reasoning", ""),
        "evidence":  evidence_list,
    }


def verify_all_claims(claims: list[str]) -> list[dict]:
    results = []
    for claim in tqdm(claims, desc="Verifying claims"):
        evidence = retrieve(claim)
        result = verify_claim(claim, evidence)
        results.append(result)
    return results


verification_results = verify_all_claims(claims)

print("\n=== Verification Results ===")
for r in verification_results:
    emoji = {"SUPPORTED": "🟢", "NOT_SUPPORTED": "🔴", "INSUFFICIENT": "🟡"}.get(r["label"], "⚪")
    print(f"{emoji} {r['label']}")
    print(f"   Claim: {r['claim']}")
    print(f"   Reason: {r['reasoning']}\n")

Verifying claims: 100%|██████████| 4/4 [00:10<00:00,  2.60s/it]


=== Verification Results ===
🟢 SUPPORTED
   Claim: У 2014 році представники Великої Британії активно підтримували суверенітет України.
   Reason: Докази підтверджують, що у 2014 році Велика Британія активно підтримувала Україну, засуджуючи дії Росії та реалізуючи санкції на підтримку українського суверенітету.

🟢 SUPPORTED
   Claim: Представники Великої Британії засуджували анексію Криму Росією.
   Reason: Докази підтверджують, що міжнародна реакція на анексію Криму Росією була негативною, зокрема уряди різних країн, включаючи Великобританію, засудили дії Російської Федерації.

🟢 SUPPORTED
   Claim: Представники Великої Британії заявляли про готовність допомогти Україні в забезпеченні її територіальної цілісності.
   Reason: Докази підтверджують, що Велика Британія активно підтримує Україну та засуджує дії Росії, що свідчить про готовність допомогти Україні в забезпеченні її територіальної цілісності.

🟢 SUPPORTED
   Claim: Представники Великої Британії закликали до міжнародної підтри

## Narrative Framing Detection

**Idea:** Ask different LLMs (GPT-4o, Gemini-simulated, neutral) the same political question.  
Then detect if they frame the same event differently — this is the "narrative framing" part of the project title.

In [ ]:
FRAMING_PERSONAS = {
    "neutral": "Ти нейтральний енциклопедичний асистент. Відповідай фактично та збалансовано.",
    "pro_western": "Ти асистент, який висвітлює події з проєвропейської та прозахідної точки зору.",
    "pro_russian": "Ти асистент, який висвітлює події з точки зору російських офіційних джерел та наративів.",
}


def generate_framed_answer(question: str, persona: str, system_prompt: str) -> str:
    """Generate an answer with a specific narrative framing."""
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user",   "content": question}
        ],
        temperature=0.3,
        max_tokens=300
    )
    return response.choices[0].message.content

framing_question = "Що відбулося під час Євромайдану 2013-2014 років?"

framed_answers = {}
for persona, system_prompt in FRAMING_PERSONAS.items():
    print(f"Generating {persona} answer...")
    framed_answers[persona] = generate_framed_answer(framing_question, persona, system_prompt)

print("\n=== Framed Answers ===")
for persona, answer in framed_answers.items():
    print(f"\n--- {persona.upper()} ---")
    print(answer)

Generating neutral answer...
Generating pro_western answer...
Generating pro_russian answer...

=== Framed Answers ===

--- NEUTRAL ---
Євромайдан, або Революція Гідності, — це серія протестів і громадських акцій, які відбулися в Україні з листопада 2013 року до лютого 2014 року. Основною причиною протестів стало рішення уряду України призупинити підписання угоди про асоціацію з Європейським Союзом, що викликало обурення серед населення, яке прагнуло євроінтеграції.

Протести розпочалися 21 листопада 2013 року на Майдані Незалежності в Києві. Спочатку акції були мирними, але з часом, особливо після розгону протестувальників у ніч на 30 листопада, ситуація загострилася. Протестувальники вимагали відставки президента Віктора Януковича, проведення реформ, боротьби з корупцією та забезпечення прав людини.

У січні 2014 року протести переросли в масові зіткнення між демонстрантами та правоохоронцями. У лютому 2014 року, після кількох днів запеклих боїв, ситуація досягла критичної точки. 18-

In [ ]:
def analyze_framing(question: str, framed_answers: dict) -> dict:
    """
    For each framed answer:
    1. Extract claims
    2. Verify against knowledge base
    3. Compare label distributions across personas
    """
    framing_analysis = {}
    
    for persona, answer in framed_answers.items():
        print(f"\nAnalyzing {persona}...")
        claims = extract_claims_zeroshot(answer)
        results = verify_all_claims(claims)
        
        label_counts = {"SUPPORTED": 0, "NOT_SUPPORTED": 0, "INSUFFICIENT": 0}
        for r in results:
            label_counts[r["label"]] = label_counts.get(r["label"], 0) + 1
        
        framing_analysis[persona] = {
            "answer":       answer,
            "claims":       claims,
            "results":      results,
            "label_counts": label_counts,
            "accuracy":     label_counts["SUPPORTED"] / max(len(claims), 1),
        }
    
    return framing_analysis


framing_analysis = analyze_framing(framing_question, framed_answers)

print("\n=== Narrative Framing Comparison ===")
print(f"Question: {framing_question}\n")
print(f"{'Persona':<15} {'✅ Supported':<15} {'❌ Not Supported':<18} {'❓ Insufficient':<15} {'Accuracy'}")
print("-" * 75)
for persona, data in framing_analysis.items():
    lc = data["label_counts"]
    print(f"{persona:<15} {lc['SUPPORTED']:<15} {lc['NOT_SUPPORTED']:<18} {lc['INSUFFICIENT']:<15} {data['accuracy']:.0%}")


Analyzing neutral...


Verifying claims: 100%|██████████| 15/15 [00:37<00:00,  2.53s/it]



Analyzing pro_western...


Verifying claims: 100%|██████████| 11/11 [00:28<00:00,  2.61s/it]



Analyzing pro_russian...


Verifying claims: 100%|██████████| 16/16 [00:32<00:00,  2.05s/it]


=== Narrative Framing Comparison ===
Question: Що відбулося під час Євромайдану 2013-2014 років?

Persona         ✅ Supported     ❌ Not Supported    ❓ Insufficient  Accuracy
---------------------------------------------------------------------------
neutral         5               0                  10              33%
pro_western     3               0                  8               27%
pro_russian     8               1                  7               50%


In [ ]:
FRAMING_DETECT_PROMPT = """Порівняй три відповіді на одне й те саме питання та визнач наративні відмінності.

Питання: {question}

Відповідь A (нейтральна):
{neutral}

Відповідь B (прозахідна):
{pro_western}

Відповідь C (проросійська):
{pro_russian}

Проаналізуй:
1. Які ключові факти кожна версія ВКЛЮЧАЄ або ВИКЛЮЧАЄ?
2. Яку мову/термінологію кожна версія використовує? (напр. "революція" vs "переворот")
3. Хто виступає "агресором" і хто "жертвою" в кожній версії?
4. Загальний висновок: наскільки сильно відрізняється фреймінг?

Відповідай українською мовою. Будь конкретним і наводь приклади з текстів."""


def detect_narrative_framing(question: str, framing_analysis: dict) -> str:
    """Use GPT-4o-mini to explicitly analyze framing differences."""
    prompt = FRAMING_DETECT_PROMPT.format(
        question=question,
        neutral=framing_analysis["neutral"]["answer"],
        pro_western=framing_analysis["pro_western"]["answer"],
        pro_russian=framing_analysis["pro_russian"]["answer"],
    )
    
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=0,
        max_tokens=800
    )
    return response.choices[0].message.content


framing_report = detect_narrative_framing(framing_question, framing_analysis)
print("=== Narrative Framing Analysis ===")
print(framing_report)

=== Narrative Framing Analysis ===
1. **Ключові факти:**
   - **Відповідь A (нейтральна):** Включає факти про початок протестів, їхні вимоги (відставка Януковича, реформи, боротьба з корупцією), а також згадує про ескалацію насильства. Відсутня чітка оцінка подій.
   - **Відповідь B (прозахідна):** Включає акцент на європейській інтеграції та демократичних реформах, підкреслює жорстокі репресії з боку влади. Відзначає, що протестувальники вимагали не лише підписання угоди, але й покращення умов життя.
   - **Відповідь C (проросійська):** Включає факти про початок протестів і їх ескалацію, але акцентує на мирному характері протестів на початку. Відзначає, що Янукович втік, але не підкреслює політичні наслідки, які можуть бути негативними для України.

2. **Мова/термінологія:**
   - **Відповідь A:** Використовує нейтральну термінологію, наприклад, "протести", "громадські акції", "ситуація загострилася". Немає емоційно забарвлених слів.
   - **Відповідь B:** Використовує терміни, які підк

## Full Pipeline (End-to-End)

In [30]:
def run_pipeline(
    text: str,
    use_qwen: bool = False,
    qwen_model=None,
    include_framing: bool = False,
    framing_question: str = None
) -> dict:
    print("Step 1: Extracting claims...")
    if use_qwen and qwen_model:
        claims = extract_claims_qwen(text, qwen_model)
    else:
        claims = extract_claims_zeroshot(text)
    print(f"  → {len(claims)} claims extracted")
    
    print("Step 2: Verifying claims...")
    results = verify_all_claims(claims)
    
    label_counts = {"SUPPORTED": 0, "NOT_SUPPORTED": 0, "INSUFFICIENT": 0}
    for r in results:
        label_counts[r["label"]] = label_counts.get(r["label"], 0) + 1
    
    print("Step 3: Generating summary...")
    summary = generate_summary(text, results, label_counts)
    
    output = {
        "input_text":   text,
        "claims":       claims,
        "results":      results,
        "label_counts": label_counts,
        "summary":      summary,
    }
    
    if include_framing and framing_question:
        print("Step 4: Running narrative framing analysis...")
        framed = {p: generate_framed_answer(framing_question, p, sp) 
                  for p, sp in FRAMING_PERSONAS.items()}
        fa = analyze_framing(framing_question, framed)
        output["framing_analysis"] = detect_narrative_framing(framing_question, fa)
        output["framing_stats"] = {p: d["label_counts"] for p, d in fa.items()}
    
    return output


def generate_summary(text: str, results: list[dict], label_counts: dict) -> str:
    results_text = ""
    for r in results:
        emoji = {"SUPPORTED": "✅", "NOT_SUPPORTED": "❌", "INSUFFICIENT": "⚠️"}.get(r["label"], "")
        results_text += f"{emoji} {r['claim']} — {r['reasoning']}\n"
    
    prompt = f"""На основі результатів верифікації тверджень склади короткий підсумок українською мовою.

Оригінальний текст: {text[:300]}...

Результати перевірки:
{results_text}

Статистика: підтверджено {label_counts['SUPPORTED']}, не підтверджено {label_counts['NOT_SUPPORTED']}, недостатньо даних {label_counts['INSUFFICIENT']}.

Напиши 2-3 речення підсумку для звичайного користувача. Вкажи які саме твердження проблематичні."""
    
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=0,
        max_tokens=200
    )
    return response.choices[0].message.content


print("Pipeline functions ready")

Pipeline functions ready


In [ ]:
example_text = """У 2014 році Велика Британія надала Україні летальну зброю та системи 
протиповітряної оборони для захисту суверенітету. Велика Британія є членом НАТО з 1949 року. 
Анексія Криму Росією у 2014 році порушила міжнародне право та Будапештський меморандум 1994 року."""

output = run_pipeline(example_text)

print("\n" + "="*60)
print("PIPELINE RESULTS")
print("="*60)
print(f"Claims found: {len(output['claims'])}")
print(f"Labels: {output['label_counts']}\n")
print("SUMMARY:")
print(output['summary'])

print("\nDETAILED RESULTS:")
for r in output['results']:
    emoji = {"SUPPORTED": "🟢", "NOT_SUPPORTED": "🔴", "INSUFFICIENT": "🟡"}.get(r["label"], "⚪")
    print(f"{emoji} [{r['label']}] {r['claim']}")
    print(f"   → {r['reasoning']}")
    if r['evidence']:
        top_e = r['evidence'][0]
        print(f"   📄 Source: {top_e['source']} (Tier {top_e['tier']})")
    print()

Step 1: Extracting claims...
  → 4 claims extracted
Step 2: Verifying claims...


Verifying claims: 100%|██████████| 4/4 [00:08<00:00,  2.17s/it]


Step 3: Generating summary...

PIPELINE RESULTS
Claims found: 4
Labels: {'SUPPORTED': 2, 'NOT_SUPPORTED': 0, 'INSUFFICIENT': 2}

SUMMARY:
У 2014 році Велика Британія надала Україні летальну зброю та системи протиповітряної оборони, але це твердження не підтверджено через відсутність конкретних доказів. Також не вдалося підтвердити інформацію про членство Великої Британії в НАТО з 1949 року. Натомість, анексія Криму Росією у 2014 році визнана порушенням міжнародного права та Будапештського меморандуму.

DETAILED RESULTS:
🟡 [INSUFFICIENT] У 2014 році Велика Британія надала Україні летальну зброю та системи протиповітряної оборони.
   → Докази не містять конкретної інформації про надання Великою Британією летальної зброї та систем протиповітряної оборони Україні у 2014 році. Вони згадують про підтримку України після вторгнення Росії в 2022 році, але не охоплюють події 2014 року.
   📄 Source: Wikipedia EN (Tier 8)

🟡 [INSUFFICIENT] Велика Британія є членом НАТО з 1949 року.
   → Наведені д

## Error Analysis

In [ ]:
TEST_CASES = [
    {
        "text": "У 2014 році Велика Британія надала Україні летальну зброю та системи ППО.",
        "expected": "NOT_SUPPORTED",  # lethal weapons only post-2022
        "note": "Temporal hallucination — weapons timeline wrong"
    },
    {
        "text": "Будапештський меморандум 1994 року гарантував Україні членство в НАТО.",
        "expected": "NOT_SUPPORTED",  # it guaranteed security assurances, not NATO membership
        "note": "Factual error — confusing security assurances with NATO membership guarantee"
    },
    {
        "text": "Голодомор 1932-1933 років був організований радянською владою і забрав мільйони життів.",
        "expected": "SUPPORTED",
        "note": "Correct historical fact"
    },
    {
        "text": "Україна офіційно вступила до НАТО у 2023 році.",
        "expected": "NOT_SUPPORTED",  # Ukraine is not a NATO member
        "note": "Hallucination — Ukraine is not a NATO member"
    },
    {
        "text": "Євромайдан 2013-2014 років розпочався через відмову Януковича підписати Угоду про асоціацію з ЄС.",
        "expected": "SUPPORTED",
        "note": "Correct — well-documented trigger"
    },
]

print("=== Error Analysis ===")
correct = 0
errors = []

for tc in TEST_CASES:
    claims = extract_claims_zeroshot(tc["text"])
    results = verify_all_claims(claims)

    predicted = results[0]["label"] if results else "INSUFFICIENT"
    is_correct = predicted == tc["expected"]
    correct += int(is_correct)
    
    status = "✅" if is_correct else "❌"
    print(f"{status} [{tc['expected']} → {predicted}] {tc['text'][:80]}")
    print(f"   Note: {tc['note']}")
    
    if not is_correct:
        errors.append({
            "text": tc["text"],
            "expected": tc["expected"],
            "predicted": predicted,
            "note": tc["note"],
            "evidence": results[0]["evidence"] if results else [],
            "reasoning": results[0]["reasoning"] if results else ""
        })
    print()

print(f"Accuracy: {correct}/{len(TEST_CASES)} = {correct/len(TEST_CASES):.0%}")

if errors:
    print("\n=== Error Cases (what went wrong) ===")
    for e in errors:
        print(f"\nClaim: {e['text']}")
        print(f"Expected: {e['expected']}, Got: {e['predicted']}")
        print(f"Reasoning: {e['reasoning']}")
        print(f"Why wrong: {e['note']}")
        if e['evidence']:
            print(f"Top evidence source: {e['evidence'][0]['source']} (Tier {e['evidence'][0]['tier']})")
            print(f"Evidence text: {e['evidence'][0]['text'][:200]}...")

=== Error Analysis ===


Verifying claims: 100%|██████████| 2/2 [00:03<00:00,  1.66s/it]


✅ [NOT_SUPPORTED → NOT_SUPPORTED] У 2014 році Велика Британія надала Україні летальну зброю та системи ППО.
   Note: Temporal hallucination — weapons timeline wrong



Verifying claims: 100%|██████████| 2/2 [00:03<00:00,  1.98s/it]


❌ [NOT_SUPPORTED → SUPPORTED] Будапештський меморандум 1994 року гарантував Україні членство в НАТО.
   Note: Factual error — confusing security assurances with NATO membership guarantee



...

❌ [SUPPORTED → INSUFFICIENT] Голодомор 1932-1933 років був організований радянською владою і забрав мільйони 
   Note: Correct historical fact



Verifying claims: 100%|██████████| 1/1 [00:01<00:00,  1.97s/it]


❌ [NOT_SUPPORTED → INSUFFICIENT] Україна офіційно вступила до НАТО у 2023 році.
   Note: Hallucination — Ukraine is not a NATO member



Verifying claims: 100%|██████████| 1/1 [00:02<00:00,  2.43s/it]

✅ [SUPPORTED → SUPPORTED] Євромайдан 2013-2014 років розпочався через відмову Януковича підписати Угоду пр
   Note: Correct — well-documented trigger

Accuracy: 2/5 = 40%

=== Error Cases (what went wrong) ===

Claim: Будапештський меморандум 1994 року гарантував Україні членство в НАТО.
Expected: NOT_SUPPORTED, Got: SUPPORTED
Reasoning: Докази підтверджують, що Будапештський меморандум був підписаний 5 грудня 1994 року.
Why wrong: Factual error — confusing security assurances with NATO membership guarantee
Top evidence source: Semantic Scholar (Tier 7)
Evidence text: The Budapest Memorandum for Ukraine (the official name is the Memorandum on Security Guarantees in Connection with Ukraine’s Accession to the Treaty on the Non-Proliferation of Nuclear Weapons) was th...

Claim: Голодомор 1932-1933 років був організований радянською владою і забрав мільйони життів.
Expected: SUPPORTED, Got: INSUFFICIENT
Reasoning: Наведені докази не містять конкретної інформації про Голодомор 1932-1933 ро

## API-ready function (for FastAPI integration)

In [ ]:
def verify_text_api(text: str) -> dict:
    claims = extract_claims_zeroshot(text)
    results_raw = verify_all_claims(claims)
    
    label_counts = {"SUPPORTED": 0, "NOT_SUPPORTED": 0, "INSUFFICIENT": 0}
    for r in results_raw:
        label_counts[r["label"]] = label_counts.get(r["label"], 0) + 1
    
    summary = generate_summary(text, results_raw, label_counts)
    
    return {
        "claims": claims,
        "results": [
            {
                "claim":     r["claim"],
                "label":     r["label"],
                "reasoning": r["reasoning"],
                "evidence":  r["evidence"][0]["text"][:300] if r["evidence"] else "",
                "source":    r["evidence"][0]["source"] if r["evidence"] else "",
                "tier":      r["evidence"][0]["tier"] if r["evidence"] else None,
            }
            for r in results_raw
        ],
        "label_counts": label_counts,
        "summary": summary,
    }

api_result = verify_text_api(example_text)
print(json.dumps(api_result, ensure_ascii=False, indent=2))

Verifying claims: 100%|██████████| 4/4 [00:08<00:00,  2.06s/it]


{
  "claims": [
    "У 2014 році Велика Британія надала Україні летальну зброю та системи протиповітряної оборони.",
    "Велика Британія є членом НАТО з 1949 року.",
    "Анексія Криму Росією у 2014 році порушила міжнародне право.",
    "Анексія Криму Росією у 2014 році порушила Будапештський меморандум 1994 року."
  ],
  "results": [
    {
      "claim": "У 2014 році Велика Британія надала Україні летальну зброю та системи протиповітряної оборони.",
      "label": "INSUFFICIENT",
      "reasoning": "Докази не містять конкретної інформації про надання Великою Британією летальної зброї та систем протиповітряної оборони Україні у 2014 році. Вони згадують про підтримку України після вторгнення Росії в 2022 році, але не охоплюють події 2014 року.",
      "evidence": "decision and praised current ongoing military training programs between both countries. Russian invasion of Ukraine Following the Russian invasion of Ukraine, the UK provided Ukraine substantial support in the form of defensi